In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/MScDS&AI/DataPreprocessing-II/employment_records.csv')
print("Dataset loaded successfully.")


Dataset loaded successfully.


In [3]:
print("\n" + "="*50)
print("Task 1: Identify Attributes / Features")
print("="*50)
print("The features in the dataset are:")
print(df.columns.tolist())


print("\n" + "="*50)
print("Task 2: Identify Feature Types")
print("="*50)
print("Data types of each feature:")
print(df.info())
print("\nSummary of Feature Types:")
print("- **Numerical:** 'Salary', 'Bonus'")
print("- **Categorical (Nominal):** 'Gender', 'Nationality', 'Marital Status'")
print("- **Categorical (Ordinal):** 'Job Title' (can be treated as ordinal)")
print("- **DateTime:** 'Date of Birth'")
print("- **Identifiers/Unique Text:** 'Full Name', 'Employee ID', 'Contact Information'")



Task 1: Identify Attributes / Features
The features in the dataset are:
['Full Name', 'Employee ID', 'Contact Information', 'Date of Birth', 'Gender', 'Nationality', 'Marital Status', 'Job Title', 'Salary', 'Bonus']

Task 2: Identify Feature Types
Data types of each feature:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Full Name            100 non-null    object 
 1   Employee ID          100 non-null    object 
 2   Contact Information  100 non-null    object 
 3   Date of Birth        100 non-null    object 
 4   Gender               100 non-null    object 
 5   Nationality          100 non-null    object 
 6   Marital Status       100 non-null    object 
 7   Job Title            100 non-null    object 
 8   Salary               100 non-null    float64
 9   Bonus                100 non-null    float64
dtypes: float64(2),

In [4]:
# Create a copy for manipulation to keep the original df intact
df_processed = df.copy()
print("Task 3: Perform Encoding")

# i. One-Hot Encoding of Gender using Scikit-learn
print("\n--- 3.i. One-Hot Encoding ---")

# Instantiate the encoder
# sparse_output=False returns a numpy array instead of a sparse matrix
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

# Reshape the 'Gender' column for the encoder and fit-transform
gender_encoded = ohe.fit_transform(df_processed[['Gender']])

# Create a new DataFrame with the encoded variables, using feature names from the encoder
# and ensuring the index aligns for concatenation
df_one_hot = pd.DataFrame(gender_encoded, columns=ohe.get_feature_names_out(['Gender']), index=df_processed.index)

# Drop the original 'Gender' column and concatenate the new one-hot encoded columns
df_processed = df_processed.drop(columns=['Gender'])
df_processed = pd.concat([df_processed, df_one_hot], axis=1)

print("Performed one-hot encoding on: ['Gender']")
print("First 5 rows of the dataset after one-hot encoding:")
# Display a few of the new columns along with an identifier
print(df_processed[['Full Name'] + list(df_one_hot.columns)].head())

# ii. Ordinal Encoding of Job title
print("\n--- 3.ii. Ordinal Encoding ---")
# Note: Treating 'Job Title' as ordinal imposes an arbitrary order.
# This is done here as requested by the user.
job_title_col = ['Job Title']
ordinal_encoder = OrdinalEncoder()

# Fit and transform the 'Job Title' column
df_processed[job_title_col] = ordinal_encoder.fit_transform(df_processed[job_title_col])

print("Performed ordinal encoding on 'Job Title'.")
print("First 5 rows of the 'Job Title' column after ordinal encoding:")
print(df_processed[['Job Title']].head())




Task 3: Perform Encoding

--- 3.i. One-Hot Encoding ---
Performed one-hot encoding on: ['Gender']
First 5 rows of the dataset after one-hot encoding:
         Full Name  Gender_Female  Gender_Male  Gender_Other
0      Trevor King            0.0          1.0           0.0
1     Jason Rhodes            1.0          0.0           0.0
2    George Tanner            0.0          0.0           1.0
3      Rita Lucero            0.0          1.0           0.0
4  Alec Washington            0.0          1.0           0.0

--- 3.ii. Ordinal Encoding ---
Performed ordinal encoding on 'Job Title'.
First 5 rows of the 'Job Title' column after ordinal encoding:
   Job Title
0        1.0
1        5.0
2        7.0
3        4.0
4        2.0


In [6]:
# ---  Perform the normalization of Salary and Bonus using Standardization ---
print("Task 4: Standardization of Salary and Bonus")

scaler = StandardScaler()
numerical_cols = ['Salary', 'Bonus']

df_processed[numerical_cols] = scaler.fit_transform(df_processed[numerical_cols])

print(f"Performed standardization (Z-score scaling) on: {numerical_cols}")
print("First 5 rows of 'Salary' and 'Bonus' after standardization:")
print(df_processed[numerical_cols].head())


Task 4: Standardization of Salary and Bonus
Performed standardization (Z-score scaling) on: ['Salary', 'Bonus']
First 5 rows of 'Salary' and 'Bonus' after standardization:
     Salary     Bonus
0 -1.344566  1.166146
1  0.818073  0.542809
2 -0.565363 -0.081282
3 -0.297081 -0.956305
4 -1.480644  0.297999


In [8]:
# --- Perform Train, Validation, Test splitting ---

print("Task 5: Train, Validation, Test Splitting")


# Define features (X) and target (y)
# Drop non-feature columns like identifiers and the original target
X = df_processed.drop(columns=['Salary', 'Full Name', 'Employee ID', 'Contact Information', 'Date of Birth'])
y = df_processed['Salary']

# Step 1: Split into training (70%) and a temporary set (30%) for validation and testing
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Step 2: Split the temporary set into validation (15%) and test (15%)
# This is 50% of the temporary set (0.5 * 0.3 = 0.15)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print("Dataset split into Train (70%), Validation (15%), and Test (15%) sets.")
print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print("-" * 20)
print(f"Shape of X_val:   {X_val.shape}")
print(f"Shape of y_val:   {y_val.shape}")
print("-" * 20)
print(f"Shape of X_test:  {X_test.shape}")
print(f"Shape of y_test:  {y_test.shape}")



Task 5: Train, Validation, Test Splitting
Dataset split into Train (70%), Validation (15%), and Test (15%) sets.
Shape of X_train: (70, 7)
Shape of y_train: (70,)
--------------------
Shape of X_val:   (15, 7)
Shape of y_val:   (15,)
--------------------
Shape of X_test:  (15, 7)
Shape of y_test:  (15,)
